# Prompt Management with LangSmith

This notebook demonstrates **prompt management** using LangSmith for a stock price assistant chatbot.

## What you will learn

| Concept | Description |
|---------|-------------|
| **Template** | Prompts in LangSmith are templates with variables (e.g. `{user_tickers}`) that get filled at runtime |
| **Messages List** | A chat prompt is a structured **list** of message templates, not a flat string |
| **Role** | Each message carries a **role** (`system`, `human`, `ai`) that shapes how the LLM interprets it |

## Project

A chatbot that checks stock prices using Yahoo Finance, personalized with the user's preferred ticker list.

## Setup

1. Run the setup cell below first — all other cells depend on it.
2. Set `MODEL_PROVIDER = "openai"` or `"gemini"`.
3. You need `LANGCHAIN_API_KEY` in your `.env` file for LangSmith access.
4. If packages are missing, run in your terminal:
   ```bash
   uv add langsmith yfinance
   ```

In [1]:
import os
import logging

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langsmith import Client
import yfinance as yf

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# ── Model selection ───────────────────────────────────────────────────────────
MODEL_PROVIDER = "openai"  # "gemini" | "openai"

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")

assert LANGCHAIN_API_KEY, "LANGCHAIN_API_KEY not found — add it to your .env file"

# ── Enable LangSmith tracing (observability for free) ──────────────────────────
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "stock-price-assistant"

# ── Shared LLM ────────────────────────────────────────────────────────────────
if MODEL_PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found — check your .env file"
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0,
    )
else:
    assert GEMINI_API_KEY, "GEMINI_API_KEY not found — check your .env file"
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash-lite",
        google_api_key=GEMINI_API_KEY,
        temperature=0,
    )

logger.info("Setup complete — LLM provider: %s", MODEL_PROVIDER)

INFO | Setup complete — LLM provider: openai


## Stock Price Tool

Define a tool that fetches current stock prices using Yahoo Finance (`yfinance`).
The agent will call this tool when the user asks about stock prices.

In [2]:
@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a given ticker symbol.

    Args:
        ticker: Stock ticker symbol (e.g., AAPL, GOOGL, TSLA)
    Returns:
        Current stock price information including daily change
    """
    logger.info("get_stock_price called | ticker=%s", ticker)
    stock = yf.Ticker(ticker)
    hist = stock.history(period="1d")
    if hist.empty:
        return f"Could not fetch price for {ticker}"
    price = hist["Close"].iloc[-1]
    prev_close = stock.info.get("previousClose")
    result = f"{ticker}: ${price:.2f}"
    if prev_close:
        change = price - prev_close
        pct = (change / prev_close) * 100
        result += f" (change: {change:+.2f}, {pct:+.2f}%)"
    logger.info("get_stock_price result: %s", result)
    return result


logger.info("Stock price tool ready")

INFO | Stock price tool ready


---

## 1. Template — Pull Prompt from LangSmith

A **prompt template** is a reusable prompt with **variables** (placeholders like `{user_tickers}`)
that get filled with real values at runtime.

In LangSmith, you create and version prompts in the UI, then pull them in code:

```python
from langsmith import Client
client = Client()
prompt = client.pull_prompt("my-prompt-name")
```

The returned object is a `ChatPromptTemplate` — LangChain's standard prompt class.

> **Prerequisite:** Create the prompt in LangSmith before running the next cell.
> Update `PROMPT_NAME` below to match your prompt's name.

In [ ]:
# ── Pull prompt from LangSmith ─────────────────────────────────────────────────
PROMPT_NAME = "stock-price-assistant"  # ← replace with your prompt name

client = Client()
prompt = None # TODO: pull the prompt from LangSmith

# The prompt is a TEMPLATE — it has placeholders that we fill at runtime
print(f"Type       : {type(prompt).__name__}")
print(f"Variables  : {prompt.input_variables}")

Type       : ChatPromptTemplate
Variables  : ['question']


---

## 2. Messages List — A Prompt is a List of Messages

Unlike a simple string prompt, a **chat prompt** is a structured **list of messages**.
Each entry in the list is a message template that becomes a real message when formatted.

```python
prompt.messages  # → [SystemMessagePromptTemplate, HumanMessagePromptTemplate, ...]
```

This matters because:
- Each message can have its own template variables
- The **order** of messages matters — the LLM reads them top to bottom
- You can mix system instructions, example conversations, and user input in one prompt

In [4]:
# ── Inspect the messages list ──────────────────────────────────────────────────
print(f"Number of messages in prompt: {len(prompt.messages)}\n")

for i, msg_template in enumerate(prompt.messages):
    cls_name = type(msg_template).__name__
    print(f"  [{i}] {cls_name}")

    if hasattr(msg_template, "prompt") and hasattr(msg_template.prompt, "template"):
        template_text = msg_template.prompt.template
        preview = template_text[:120] + ("..." if len(template_text) > 120 else "")
        print(f"      Template: \"{preview}\"")
    elif hasattr(msg_template, "variable_name"):
        print(f"      Placeholder variable: {msg_template.variable_name}")
    print()

Number of messages in prompt: 2

  [0] SystemMessagePromptTemplate
      Template: "Answer user's question"

  [1] HumanMessagePromptTemplate
      Template: "{question}"



---

## 3. Roles — Each Message Has a Role

Every message in a chat prompt has a **role** that tells the LLM how to interpret it:

| Role | Purpose | Example |
|------|---------|--------|
| **system** | Sets the agent's behavior, personality, and constraints | *"You are a stock assistant…"* |
| **human** | Represents user input | *"What's the price of AAPL?"* |
| **ai** | Represents assistant responses (useful for few-shot examples) | *"AAPL is currently at $150.00"* |

The **system** role is especially powerful — it shapes the entire conversation.
This is where we inject the user's preferred tickers via the `{user_tickers}` template variable.

In [5]:
# ── Show the role of each message ──────────────────────────────────────────────
ROLE_MAP = {
    "SystemMessagePromptTemplate": "system",
    "HumanMessagePromptTemplate": "human",
    "AIMessagePromptTemplate": "ai",
    "MessagesPlaceholder": "placeholder",
}

print("Message roles in this prompt:\n")
for i, msg_template in enumerate(prompt.messages):
    cls_name = type(msg_template).__name__
    role = ROLE_MAP.get(cls_name, cls_name)
    print(f"  [{i}] Role: {role:<12} | Type: {cls_name}")

Message roles in this prompt:

  [0] Role: system       | Type: SystemMessagePromptTemplate
  [1] Role: human        | Type: HumanMessagePromptTemplate


---

## Format the Template

Now we **format** the entire template at once using `prompt.format_messages()`.
This fills in **all** variables across every message and returns a list of concrete
`BaseMessage` objects (`SystemMessage`, `HumanMessage`, etc.) — ready to send to the LLM.

```python
compiled = prompt.format_messages(user_tickers="AAPL, GOOGL", question="How are my stocks?")
# → [SystemMessage(content="..."), HumanMessage(content="...")]
```

This is cleaner than formatting individual messages via `prompt.messages[0].format(...)` —
you get the full compiled message list in one call.

In [ ]:
# ── User's preferred tickers ───────────────────────────────────────────────────
USER_TICKERS = ["AAPL", "GOOGL", "TSLA"]

compiled_messages = None # TODO: format the prompt with the user's tickers to create a compiled list of messages

print(f"Number of compiled messages: {len(compiled_messages)}\n")
for i, msg in enumerate(compiled_messages):
    print(f"  [{i}] Type: {type(msg).__name__}")
    print(f"      Role: {msg.type}")
    print(f"      Content: {msg.content}")
    print()

Number of compiled messages: 2

  [0] Type: SystemMessage
      Role: system
      Content: Answer user's question

  [1] Type: HumanMessage
      Role: human
      Content: How are my preferred stocks doing today?



---

## Build the Agent

Wire everything together:
1. The **formatted system prompt** (with tickers injected) tells the agent who it is
2. The **stock price tool** gives the agent the ability to look up real prices
3. `create_agent` builds a ReAct loop that reasons and acts autonomously

In [ ]:
# ── Extract system content from compiled messages ─────────────────────────────
system_content = next(msg.content for msg in compiled_messages if msg.type == "system")

# ── Build the agent with the formatted prompt ─────────────────────────────────
agent = create_agent(
    model=llm,
    tools=[get_stock_price],
    # system_prompt=system_content, # if we provide a system prompt here, we need to provide ai/human messages only later
)

logger.info("Agent created — prompt injected, tools: %s", [get_stock_price.name])

INFO | Agent created — prompt injected, tools: ['get_stock_price']


---

## Test the Agent

Ask the agent about the user's preferred stocks.
Because the system prompt contains the ticker list, the agent knows which stocks to check.

In [ ]:
from langchain_core.messages import messages_to_dict
# ── Test 1: Ask about preferred stocks ─────────────────────────────────────────
result = None # TODO: invoke the agent with the compiled messages

print(result["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Please provide me with the ticker symbols of your preferred stocks, and I'll check their current stock prices for you.


In [ ]:
# ── Test 2: Ask about a specific stock ─────────────────────────────────────────
result2 = None # TODO: invoke the agent with a specific stock

print(result2["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO | get_stock_price called | ticker=NVDA
INFO | get_stock_price result: NVDA: $167.52 (change: -3.72, -2.17%)
INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The current price of NVDA (NVIDIA Corporation) is $167.52, with a change of -$3.72, which is a decrease of 2.17%.


WARNING | Failed to refresh cache entry stock-price-assistant: Connection error caused failure to GET /commits/-/stock-price-assistant/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/-/stock-price-assistant/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: None
API Key: lsv2_********************************************7c


---

## Passing Chat History via a Messages Placeholder

LangSmith prompts support a **Messages Placeholder** — a slot in the prompt template
where a list of messages (e.g. conversation history) gets inserted at runtime.

When pulled with `client.pull_prompt()`, the placeholder appears as a
`MessagesPlaceholder` in the `ChatPromptTemplate`. You then pass the history
directly to `format_messages()` alongside your other variables:

```python
compiled = prompt.format_messages(
    user_tickers="AAPL, GOOGL",
    question="What is the price?",
    chat_history=[HumanMessage(...), AIMessage(...), ...],
)
```

> **Prerequisite:** Add a *Messages Placeholder* named `chat_history` to your
> `stock-price-assistant` prompt in the LangSmith UI, then re-run from here.

In [ ]:
# ── Re-fetch the prompt (now with the chat_history placeholder) ───────────────
from langchain_core.messages import AIMessage

prompt_with_ph = client.pull_prompt(PROMPT_NAME)

logger.info("Prompt messages after adding placeholder:")
for i, msg_template in enumerate(prompt_with_ph.messages):
    cls_name = type(msg_template).__name__
    logger.info("  [%d] %s", i, cls_name)

# ── Format the entire prompt with chat history in one call ────────────────────
compiled_with_history = None # TODO: format the prompt with the chat history to create a compiled list of messages

logger.info("Compiled messages with chat history:")
for i, msg in enumerate(compiled_with_history):
    preview = msg.content[:100] + ("..." if len(msg.content) > 100 else "")
    logger.info("  [%d] %-6s | %s", i, msg.type, preview)

INFO | Prompt messages after adding placeholder:
INFO |   [0] SystemMessagePromptTemplate
INFO |   [1] HumanMessagePromptTemplate
INFO | Compiled messages with chat history:
INFO |   [0] system | Answer user's question
INFO |   [1] human  | What is the current price of it


In [18]:
# ── Run the agent with the compiled messages ──────────────────────────────────
result_hist = agent.invoke({"messages": compiled_with_history})
print(result_hist["messages"][-1].content)

INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Could you please specify the stock ticker symbol for the company you are interested in?


---

## Using Model Config from a Pulled Prompt

When you create a prompt in the LangSmith UI, you can also set **model configuration**
(model name, temperature, max tokens, etc.) alongside the template.

Use `include_model=True` when pulling the prompt to get a **runnable chain**
(`prompt | model`) with those settings baked in:

```python
chain = client.pull_prompt("my-prompt", include_model=True)
# → RunnableSequence(ChatPromptTemplate | ChatOpenAI(temperature=0.7, ...))
```

This is useful when you want **non-developers** (e.g. prompt engineers) to control
model parameters directly from the LangSmith UI — no code changes needed.

In [ ]:
# ── Pull prompt WITH model config from LangSmith ──────────────────────────────
chain = client.pull_prompt(PROMPT_NAME, include_model=True)

print(f"Returned type: {type(chain).__name__}")
print(f"Chain: {chain}\n")

# The chain is a RunnableSequence: prompt_template | configured_model
prompt_part = chain.first
model_part = chain.last

print(f"Prompt : {type(prompt_part).__name__}")
print(f"Model  : {type(model_part).__name__}")
print(f"  model_name  : {model_part.model_name}")
print(f"  temperature : {model_part.temperature}")

if hasattr(model_part, "max_tokens") and model_part.max_tokens is not None:
    print(f"  max_tokens  : {model_part.max_tokens}")

# ── Use the config values programmatically ─────────────────────────────────────
# e.g. override just the temperature from the pulled config
pulled_temperature = model_part.temperature
logger.info("Using temperature=%.2f from LangSmith prompt config", pulled_temperature)

if MODEL_PROVIDER == "openai":
    llm_from_config = ChatOpenAI(
        model=model_part.model_name,
        api_key=OPENAI_API_KEY,
        temperature=pulled_temperature,
    )
else:
    llm_from_config = ChatGoogleGenerativeAI(
        model=model_part.model_name,
        google_api_key=GEMINI_API_KEY,
        temperature=pulled_temperature,
    )

# ── Invoke the chain directly (prompt + model in one call) ────────────────────
response = chain.invoke({"question": "What is AAPL?"})
print(f"\nDirect chain response:\n{response.content}")

---

## Wrap-up

### What we covered

| Concept | What you saw |
|---------|-------------|
| **Template** | `prompt.input_variables` — the prompt has `{user_tickers}` filled at runtime |
| **Messages List** | `prompt.messages` — a prompt is a list of message templates, not a flat string |
| **Role** | Each message has a role (`system`, `human`, `ai`) that shapes LLM behavior |
| **Placeholder** | `MessagesPlaceholder("chat_history")` — inject conversation history into the prompt via `format_messages()` |

### Key takeaways

- **LangSmith** stores prompts centrally — update them in the UI without redeploying code
- **Templates** let you inject user-specific data (like preferred tickers) at runtime
- **Placeholders** let you inject conversation history into a prompt template via `MessagesPlaceholder`
- **Roles** control how the LLM interprets each message — the `system` role is the most powerful lever
- **Observability** is automatic — every agent run is traced in LangSmith (check your project dashboard)